# Стоит ли ПРАЙМ стимулировать совместное использование подписки?

Связано ли добавление участников в подписку с более высоким удержанием владельца и большей ценностью подписки, и есть ли смысл продуктово стимулировать шеринг?

Подписки с участниками более длительные - их реже отменяют, они дольше живут и приносят больше выручки. Если эффект устойчив внутри тарифов и не объясняется только возрастом подписки, ПРАЙМ стоит стимулировать владельцев подключать участников.

Основное зерно анализа оттока (подписка и 30-дневный период жизни). Такое зерно позволяет понять на каком месяце жизни подписки происходит отток 

# Подключение данных

In [ ]:
import json
import sys

import numpy as np
import pandas as pd
from sqlalchemy import text

# get_engine() живёт в вашем репозитории, в src/prime/config.py.
# Она читает строку подключения из .env — поэтому пароля здесь нет.
try:
    from prime.config import get_engine
except ModuleNotFoundError:
    raise ModuleNotFoundError(
        "Пакет prime не найден: ноутбук запущен не в окружении prime-monitor.\n"
        f"Сейчас Python отсюда: {sys.executable}\n"
        "JupyterLab: закройте его, перейдите в корень prime-monitor и выполните uv run jupyter lab.\n"
        "VS Code: Select Kernel справа сверху -> .venv из папки prime-monitor."
    ) from None

engine = get_engine()


def q(sql: str, **params) -> pd.DataFrame:
    """Выполнить запрос и вернуть DataFrame. Соединение закрывается сразу."""
    try:
        with engine.connect() as conn:
            # Моменты времени считаем в UTC: так они у всех одинаковые.
            conn.execute(text("set time zone 'UTC'"))
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        engine.dispose()


print("подключились как:", q("select current_user as login").iloc[0, 0])

# Проверка таблиц

In [ ]:
subs = q("""
    select
        subscription_id
        , owner_client_id
        , tariff_id
        , started_at
        , ended_at
        , cancel_reason
    from prime.subscriptions
    order by subscription_id
""")

In [ ]:
members = q("""
    select
        subscription_id
        , client_id
        , joined_at
        , left_at
    from prime.subscription_members
    order by subscription_id, joined_at
""")

In [ ]:
subs.head()

In [ ]:
members.head()

In [ ]:
print("subs:", len(subs))
print("members:", len(members))

In [ ]:
members.groupby("subscription_id").size().describe()

In [ ]:
# один и тот же человек не записан дважды в одну подписку?
members.duplicated(
    subset=["subscription_id", "client_id"]
).sum()

In [ ]:
# участник не мог присоединиться раньше старта подписки
check = members.merge(
    subs[["subscription_id", "started_at"]],
    on="subscription_id",
    how="left",
    validate="m:1"
)

(check["joined_at"] < check["started_at"]).sum()

In [ ]:
# распределение: сколько подписок имеют 0 - 5 участников
n_members = (
    members.groupby("subscription_id")
    .size()
    .rename("n_members")
)

subs["subscription_id"].map(n_members).fillna(0).value_counts().sort_index()

In [ ]:
# на всякий случай проверка на то что выход не может быть раньше входа
(
    members["left_at"].notna()
    & (members["left_at"] < members["joined_at"])
).sum()

In [ ]:
print("joined:", members["joined_at"].min(), "—", members["joined_at"].max())
print("left:", members["left_at"].min(), "—", members["left_at"].max())

print("ended subs:", subs["ended_at"].min(), "—", subs["ended_at"].max())

## Проверка на то, что нет клиентов, кто подключился к подписке после ее окончания

In [ ]:
member_check = members.merge(
    subs[["subscription_id", "ended_at"]],
    on="subscription_id",
    how="left",
    validate="m:1"
)

(
    member_check["ended_at"].notna()
    & (member_check["joined_at"] > member_check["ended_at"])
).sum()

In [ ]:
bad = q("""
    select
        sm.subscription_id
        , sm.client_id
        , sm.joined_at
        , sm.left_at
        , s.started_at
        , s.ended_at
        , extract(epoch from (sm.joined_at - s.ended_at)) / 86400 as days_after_end
    from prime.subscription_members sm
    join prime.subscriptions s
        on sm.subscription_id = s.subscription_id
    where 1=1
    and s.ended_at is not null
    and sm.joined_at > s.ended_at
""")

print("Аномальных membership", len(bad))
print("Затронуто подписок", bad["subscription_id"].nunique())

bad["days_after_end"].describe()

In [ ]:
join_offsets = q("""
    select
        round(extract(epoch from (sm.joined_at - s.started_at)) / 86400)::int as join_after_start_days
        , count(*) as memberships
    from prime.subscription_members sm
    join prime.subscriptions s
        on sm.subscription_id = s.subscription_id
    where 1=1
    group by round(extract(epoch from (sm.joined_at - s.started_at)) / 86400)::int
    order by join_after_start_days
""")

join_offsets

In [ ]:
print("Минимальный offset - ", join_offsets["join_after_start_days"].min())
print("Максимальный offset - ", join_offsets["join_after_start_days"].max())
print("Частота по дням")
print("min - ", join_offsets["memberships"].min())
print("max - ", join_offsets["memberships"].max())
print("mean - ", round(join_offsets["memberships"].mean(), 1))

In [ ]:
bad_by_lifetime = q("""
    with member_lifetime as 
    (
        select
            sm.subscription_id
            , sm.client_id
            , floor(
                extract(epoch from (s.ended_at - s.started_at)) / 86400
            )::int as lifetime_days
            , case
                when sm.joined_at > s.ended_at then 1
                else 0
              end as is_bad
        from prime.subscription_members sm
        join prime.subscriptions s
            on sm.subscription_id = s.subscription_id
        where 1=1
        and s.ended_at is not null
    )
    select
        lifetime_days
        , count(*) as memberships
        , sum(is_bad) as bad
        , avg(is_bad::numeric) as bad_share
        , greatest(
            (59 - lifetime_days) / 60.0
            , 0
          ) as expected_bad_share
    from member_lifetime
    where 1=1
    group by lifetime_days
    order by lifetime_days
""")

bad_by_lifetime.head(60)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))

plt.bar(
    join_offsets["join_after_start_days"],
    join_offsets["memberships"]
)

plt.axhline(
    join_offsets["memberships"].mean(),
    linestyle="--",
    label="Среднее"
)

plt.title("Распределение даты подключения участника")
plt.xlabel("Дней после старта подписки")
plt.ylabel("Количество membership")
plt.legend()

plt.show()

Проверка качества данных. В subscription_members обнаружено 13 421 участий в 8 346 подписках, где joined_at находится после subscriptions.ended_at. Дополнительно посмотрел, что дата подключения участника практически равномерно распределена в диапазоне 0–59 дней после начала подписки и формируется независимо от даты её окончания. Дальше следует исключить таких пользователей и рассматривать гипотезу без них. 

In [ ]:
plt.figure(figsize=(10, 4))

plt.plot(
    bad_by_lifetime["lifetime_days"],
    bad_by_lifetime["bad_share"],
    marker="o",
    label="Факт"
)

plt.plot(
    bad_by_lifetime["lifetime_days"],
    bad_by_lifetime["expected_bad_share"],
    linestyle="--",
    label="Ожидание"
)

plt.xlim(0, 59)
plt.ylim(0, 1)

plt.title("Аномальные membership и срок жизни подписки")
plt.xlabel("Срок жизни подписки, дней")
plt.ylabel("Доля joined_at после ended_at")
plt.legend()

plt.show()

# Проверка гипотезы 1

## Сборка скорректированных данных

In [ ]:
panel = q("""
    with months as 
    (
        select
            generate_series(
                '2025-01-01'::timestamptz
                , '2026-07-01'::timestamptz
                , interval '1 month'
            ) as month_start
    )
    , active_subs as 
    (
        select
            m.month_start
            , s.subscription_id
            , s.tariff_id
            , s.started_at
            , s.ended_at
        from months m
        inner join prime.subscriptions s
            on s.started_at <= m.month_start
            and 
            (
                s.ended_at is null
                or s.ended_at > m.month_start
            )
    )
    , member_counts as 
    (
        select
            a.month_start
            , a.subscription_id
            , count(sm.client_id) as n_members
        from active_subs a
        left join prime.subscription_members sm
            on sm.subscription_id = a.subscription_id
            and sm.joined_at <= a.month_start
            and 
            (
                sm.left_at is null
                or sm.left_at > a.month_start
            )
        where 1=1
        group by a.month_start, a.subscription_id
    )
    select
        a.month_start
        , a.subscription_id
        , a.tariff_id
        , mc.n_members
        , case
            when a.ended_at >= a.month_start
            and a.ended_at < a.month_start + interval '1 month'
            then 1
            else 0
          end as churned
          , extract(
                epoch from (a.month_start - a.started_at)
            ) / 86400 as age_days
    from active_subs a
    join member_counts mc
        on a.month_start = mc.month_start
        and a.subscription_id = mc.subscription_id
    where 1=1
    order by a.month_start, a.subscription_id
""")

In [ ]:
panel["sharing_group"] = pd.cut(
    panel["n_members"]
    , bins=[-1, 0, 1, 2, float("inf")]
    , labels=["0", "1", "2", "3+"]
)

In [ ]:
panel["age_group"] = pd.cut(
    panel["age_days"]
    , bins=[0, 90, 180, 365, float("inf")]
    , labels=["0–3 мес.", "3–6 мес.", "6–12 мес.", "12+ мес."]
)



In [ ]:
age_result = (
    panel
    .groupby(
        ["age_group", "sharing_group"]
        , observed=True
    )
    .agg(
        observations=("subscription_id", "size")
        , churn=("churned", "mean")
    )
)

age_result["churn"] *= 100

age_result

In [ ]:
sharing_result = (
    panel
    .groupby("sharing_group", observed=True)
    .agg(
        observations=("subscription_id", "size")
        , subscriptions=("subscription_id", "nunique")
        , churned=("churned", "sum")
        , churn=("churned", "mean")
    )
)

sharing_result["churn"] *= 100

sharing_result

In [ ]:
tariff_result = (
    panel
    .groupby(
        ["tariff_id", "sharing_group"]
        , observed=True
    )
    .agg(
        observations=("subscription_id", "size")
        , churn=("churned", "mean")
    )
)

tariff_result["churn"] *= 100

tariff_result

## Вывод по гипотезе 1

Гипотеза не подтвердилась. Общий отток составляет около 4% независимо от числа участников. Результат сохраняется отдельно внутри тарифов и внутри групп по возрасту подписки. Следовательно, стимулировать шеринг именно как инструмент снижения оттока оснований нет.

# Гипотеза 2 - Владельцы подписок, кто пользуется сервисом в первые 30 дней, имею отток в следующие 60 дней меньше чем те, кто не попользовался

## Проверка данных

In [ ]:
usage = q("""
    select *
    from prime.service_usage
    limit 5
""")

usage

In [ ]:
usage_info = q("""
    select
        min(started_at) as min_usage_at
        , max(started_at) as max_usage_at
        , count(*) as usages
        , count(distinct client_id) as clients
        , count(distinct service_id) as services
    from prime.service_usage
    where 1=1
""")

usage_info

In [ ]:
usage_services = q("""
    select
        service_id
        , count(*) as usages
        , count(distinct client_id) as clients
    from prime.service_usage
    where 1=1
    group by service_id
    order by usages desc
""")

usage_services

In [ ]:
usage_devices = q("""
    select
        device_type
        , count(*) as usages
    from prime.service_usage
    where 1=1
    group by device_type
    order by usages desc
""")

usage_devices

In [ ]:
usage_mart = q("""
with eligible_subs as 
(
    select
        s.subscription_id
        , s.owner_client_id
        , s.tariff_id
        , s.started_at
        , s.ended_at
    from prime.subscriptions s
    where 1=1
    and s.started_at >= '2025-01-06'::timestamptz
    and s.started_at + interval '90 day' <= '2026-08-29'::timestamptz
    and 
    (
        s.ended_at is null
        or s.ended_at >= s.started_at + interval '30 day'
    )
)
, usage_30 as 
(
    select
        s.subscription_id
        , count(u.usage_id) as n_usages_30
        , count(distinct u.service_id) as n_services_30
    from eligible_subs s
    left join prime.service_usage u
        on u.client_id = s.owner_client_id
        and u.started_at >= s.started_at
        and u.started_at < s.started_at + interval '30 day'
    where 1=1
    group by s.subscription_id
)
select
    s.subscription_id
    , s.tariff_id
    , s.started_at
    , u.n_usages_30
    , u.n_services_30
    , case
        when u.n_usages_30 > 0 then 1
        else 0
      end as used_30
    , case
        when s.ended_at >= s.started_at + interval '30 day'
        and s.ended_at < s.started_at + interval '90 day'
        then 1
        else 0
      end as churn_31_90
from eligible_subs s
join usage_30 u
    on s.subscription_id = u.subscription_id
where 1=1
order by s.subscription_id
""")

In [ ]:
print("Строк:", len(usage_mart))
print(
    "Уникальных подписок:",
    usage_mart["subscription_id"].nunique()
)

usage_mart.head()

In [ ]:
usage_result = (
    usage_mart
    .groupby("used_30")
    .agg(
        subscriptions=("subscription_id", "size")
        , churn=("churn_31_90", "mean")
        , avg_usages=("n_usages_30", "mean")
        , avg_services=("n_services_30", "mean")
    )
)

usage_result["churn"] *= 100

usage_result

## Вывод по гипотезе 2

Использование хотя бы одного сервиса в первые 30 дней не связано со снижением оттока. Гипотеза о том, что использование в первые 30 дней подписки повышает удержание, не подтвердилась.

In [ ]:
usage_mart["usage_group"] = pd.cut(
    usage_mart["n_usages_30"]
    , bins=[-1, 0, 1, 3, 5, float("inf")]
    , labels=["0", "1", "2–3", "4–5", "6+"]
)

usage_intensity = (
    usage_mart
    .groupby("usage_group", observed=True)
    .agg(
        subscriptions=("subscription_id", "size")
        , churn=("churn_31_90", "mean")
    )
)

usage_intensity["churn"] *= 100

usage_intensity

In [ ]:
usage_mart["services_group"] = pd.cut(
    usage_mart["n_services_30"]
    , bins=[-1, 0, 1, 2, float("inf")]
    , labels=["0", "1", "2", "3+"]
)

services_result = (
    usage_mart
    .groupby("services_group", observed=True)
    .agg(
        subscriptions=("subscription_id", "size")
        , churn=("churn_31_90", "mean")
    )
)

services_result["churn"] *= 100

services_result

# Гипотеза 3 - Повышенный отток сосредоточен в первые месяцы жизни подписки и связан с конкретными причинами отмены

In [ ]:
lifecycle = q("""
with params as 
(
    select
        '2026-08-30 00:00:00+00'::timestamptz as observation_end
)
, subscription_periods as 
(
    select
        s.subscription_id
        , s.tariff_id
        , s.started_at
        , s.ended_at
        , g.period_n
        , s.started_at + g.period_n * interval '30 day' as period_start
        , s.started_at + (g.period_n + 1) * interval '30 day' as period_end
    from prime.subscriptions s
    cross join lateral generate_series(0, 40) as g(period_n)
    cross join params p
    where 1=1
    and s.started_at + (g.period_n + 1) * interval '30 day' <= p.observation_end
    and (
        s.ended_at is null
        or s.ended_at > s.started_at + g.period_n * interval '30 day'
    )
)
select
    subscription_id
    , tariff_id
    , period_n + 1 as life_month
    , period_start
    , period_end
    , case
        when ended_at > period_start
        and ended_at <= period_end
        then 1
        else 0
      end as churned
from subscription_periods
where 1=1
order by subscription_id, life_month
""")

In [ ]:
print("Строк:", len(lifecycle))

print(
    "Дублей subscription × life_month:",
    lifecycle.duplicated(
        ["subscription_id", "life_month"]
    ).sum()
)

lifecycle.head()

In [ ]:
lifecycle_result = (
    lifecycle
    .groupby("life_month")
    .agg(
        observations=("subscription_id", "size")
        , churned=("churned", "sum")
        , churn=("churned", "mean")
    )
)

lifecycle_result["churn"] *= 100

lifecycle_result.head(20)

In [ ]:
lifecycle["life_month_group"] = lifecycle["life_month"].clip(upper=18)

lifecycle_plot = (
    lifecycle
    .groupby("life_month_group")
    .agg(
        observations=("subscription_id", "size")
        , churn=("churned", "mean")
    )
    .reset_index()
)

lifecycle_plot["churn"] *= 100

lifecycle_plot["life_month_label"] = (
    lifecycle_plot["life_month_group"]
    .astype(str)
)

lifecycle_plot.loc[
    lifecycle_plot["life_month_group"] == 18,
    "life_month_label"
] = "18+"

lifecycle_plot

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))

plt.plot(
    lifecycle_plot["life_month_label"]
    , lifecycle_plot["churn"]
    , marker="o"
)

plt.title("Отток по месяцу жизни подписки")
plt.xlabel("Месяц жизни подписки")
plt.ylabel("Отток, %")

plt.show()

In [ ]:
reason_lifecycle = q("""
with ended as 
(
    select
        subscription_id
        , cancel_reason
        , ceil(
            extract(epoch from (ended_at - started_at)) / 86400 / 30.0
          )::int as life_month
    from prime.subscriptions
    where 1=1
    and ended_at is not null
)
select
    case
        when life_month between 1 and 3 then '1–3'
        when life_month between 4 and 6 then '4–6'
        when life_month between 7 and 12 then '7–12'
        when life_month between 13 and 17 then '13–17'
    end as life_group
    , cancel_reason
    , count(*) as cancellations
from ended
where 1=1
and life_month between 1 and 17
group by life_group, cancel_reason
order by life_group, cancellations desc
""")

reason_lifecycle

In [ ]:
reason_share = reason_lifecycle.copy()

reason_share["share"] = (
    reason_share["cancellations"]
    / reason_share.groupby("life_group")["cancellations"].transform("sum")
)

life_order = ["1–3", "4–6", "7–12", "13–17"]
reason_order = [
    "not_using",
    "too_expensive",
    "switched",
    "payment_failed",
    "other"
]

reason_share["life_group"] = pd.Categorical(
    reason_share["life_group"],
    categories=life_order,
    ordered=True
)

reason_share["cancel_reason"] = pd.Categorical(
    reason_share["cancel_reason"],
    categories=reason_order,
    ordered=True
)

plot_df = (
    reason_share
    .pivot(index="life_group", columns="cancel_reason", values="share")
    .fillna(0)
    .sort_index()
)

ax = plot_df.plot(
    kind="bar",
    stacked=True,
    figsize=(10, 6)
)

ax.set_title("Структура оттока по life group")
ax.set_xlabel("Группа по сроку жизни подписки")
ax.set_ylabel("Доля отмен")
ax.legend(title="Причина отмены", bbox_to_anchor=(1.05, 1), loc="upper left")

plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Вывод по гипотезе 3

Структура причин оттока практически не меняется по мере жизни подписки. Во всех группах основная причина это not_using (около 34%), далее идут too_expensive (~24%), switched (~18%) и payment_failed (~14%). Получается, что более высокий отток на ранних этапах жизни подписки не объясняется какой-то конкретной причиной отмены.

In [ ]:
tariff_lifecycle = (
    lifecycle[
        lifecycle["life_month"] <= 17
    ]
    .groupby(
        ["life_month", "tariff_id"]
    )
    .agg(
        observations=("subscription_id", "size")
        , churn=("churned", "mean")
    )
    .reset_index()
)

tariff_lifecycle["churn"] *= 100

tariff_lifecycle.head(20)

In [ ]:
plt.figure(figsize=(10, 5))

for tariff_id, data in tariff_lifecycle.groupby("tariff_id"):
    plt.plot(
        data["life_month"]
        , data["churn"]
        , marker="o"
        , label=f"Тариф {tariff_id}"
    )

plt.title("Отток по месяцу жизни и тарифу")
plt.xlabel("Месяц жизни подписки")
plt.ylabel("Отток, %")
plt.legend()
plt.show()

# Гипотеза 4 - Каналы привлечения отличаются по качеству клиентов - часть каналов приводит подписчиков с существенно более высоким оттоком в первые месяцы жизни подписки

Какие каналы приводят наиболее устойчивых подписчиков ПРАЙМ, а какие создают повышенный ранний отток, и стоит ли перераспределять маркетинговые расходы?

## Подготовка данных

In [ ]:
channel_lifecycle = q("""
with params as 
(
    select '2026-08-30 00:00:00+00'::timestamptz as observation_end
)
, subscription_periods as (
    select
        s.subscription_id
        , c.acquisition_channel
        , g.period_n + 1 as life_month
        , s.started_at + g.period_n * interval '30 day' as period_start
        , s.started_at + (g.period_n + 1) * interval '30 day' as period_end
        , s.ended_at
    from prime.subscriptions s
    join prime.clients c
        on s.owner_client_id = c.client_id
    cross join lateral generate_series(0, 16) as g(period_n)
    cross join params p
    where 1=1
    and s.started_at + (g.period_n + 1) * interval '30 day' <= p.observation_end
    and 
    (
        s.ended_at is null
        or s.ended_at > s.started_at + g.period_n * interval '30 day'
    )
)
select
    life_month
    , acquisition_channel
    , count(*) as observations
    , avg(
        case
            when ended_at > period_start
            and ended_at <= period_end
            then 1.0
            else 0.0
        end
      ) * 100 as churn
from subscription_periods
where 1=1
group by life_month, acquisition_channel
order by life_month, acquisition_channel
""")

channel_lifecycle.head(20)

In [ ]:
early_channel = (
    channel_lifecycle[
        channel_lifecycle["life_month"].between(1, 3)
    ]
    .groupby("acquisition_channel")
    .apply(
        lambda x: pd.Series({
            "observations": x["observations"].sum(),
            "churn": (
                (x["churn"] * x["observations"]).sum()
                / x["observations"].sum()
            )
        }),
        include_groups=False
    )
    .sort_values("churn", ascending=False)
)

early_channel

In [ ]:
channel_tariff = q("""
with params as 
(
    select '2026-08-30 00:00:00+00'::timestamptz as observation_end
)
, subscription_periods as 
(
    select
        s.subscription_id
        , s.tariff_id
        , c.acquisition_channel
        , g.period_n + 1 as life_month
        , s.started_at + g.period_n * interval '30 day' as period_start
        , s.started_at + (g.period_n + 1) * interval '30 day' as period_end
        , s.ended_at
    from prime.subscriptions s
    join prime.clients c
        on s.owner_client_id = c.client_id
    cross join lateral generate_series(0, 2) as g(period_n)
    cross join params p
    where 1=1
    and s.started_at + (g.period_n + 1) * interval '30 day' <= p.observation_end
    and 
    (
        s.ended_at is null
        or s.ended_at > s.started_at + g.period_n * interval '30 day'
    )
)
select
    acquisition_channel
    , tariff_id
    , count(*) as observations
    , avg(
        case
            when ended_at > period_start
            and ended_at <= period_end
            then 1.0
            else 0.0
        end
      ) * 100 as churn
from subscription_periods
where 1=1
group by acquisition_channel, tariff_id
order by acquisition_channel, tariff_id
""")

channel_tariff

Различия раннего оттока связаны прежде всего с каналом привлечения, а не с тарифом. social приводит клиентов с самым высоким ранним оттоком (~7%), organic — с самым низким (~2,2%). Разрыв превышает 4,8 п.п., или более чем в 3 раза относительно organic.

In [ ]:
channel_cohort = q("""
with params as 
(
    select '2026-08-30 00:00:00+00'::timestamptz as observation_end
)
, subscription_periods as 
(
    select
        s.subscription_id
        , c.acquisition_channel
        , date_trunc('quarter', s.started_at)::date as start_quarter
        , g.period_n + 1 as life_month
        , s.started_at + g.period_n * interval '30 day' as period_start
        , s.started_at + (g.period_n + 1) * interval '30 day' as period_end
        , s.ended_at
    from prime.subscriptions s
    join prime.clients c
        on s.owner_client_id = c.client_id
    cross join lateral generate_series(0, 2) as g(period_n)
    cross join params p
    where 1=1
    and s.started_at + (g.period_n + 1) * interval '30 day' <= p.observation_end
    and (
        s.ended_at is null
        or s.ended_at > s.started_at + g.period_n * interval '30 day'
    )
)
select
    start_quarter
    , acquisition_channel
    , count(*) as observations
    , avg(
        case
            when ended_at > period_start
            and ended_at <= period_end
            then 1.0
            else 0.0
        end
      ) * 100 as churn
from subscription_periods
where 1=1
group by start_quarter, acquisition_channel
order by start_quarter, acquisition_channel
""")

channel_cohort

In [ ]:
channel_cohort.pivot(
    index="start_quarter"
    , columns="acquisition_channel"
    , values="churn"
)

In [ ]:
plot_df = (
    early_channel
    .reset_index()
    .sort_values("churn", ascending=False)
    .reset_index(drop=True)
)

fig, ax = plt.subplots(figsize=(9, 5))

bars = ax.barh(
    plot_df["acquisition_channel"]
    , plot_df["churn"]
)

ax.invert_yaxis()

ax.bar_label(
    bars
    , labels=[f"{x:.2f}%" for x in plot_df["churn"]]
    , padding=5
)

avg_churn = (
    (plot_df["churn"] * plot_df["observations"]).sum()
    / plot_df["observations"].sum()
)

ax.axvline(
    avg_churn
    , linestyle="--"
    , label=f"Средний отток: {avg_churn:.2f}%"
)

ax.set_title("Ранний отток по каналам привлечения")
ax.set_xlabel("Отток в первые 3 месяца, %")
ax.set_ylabel("Канал привлечения")
ax.set_xlim(0, plot_df["churn"].max() + 0.8)

ax.legend()

plt.tight_layout()
plt.show()


## Вывод по гипотезе 4

Отток в первые 3 месяца существенно различается между каналами привлечения. Канал social показывает максимальный отток - 7,01%, что более чем в три раза выше organic (2,20%) и на 2,71 п.п. выше среднего по рассматриваемым каналам (4,30%). Выше среднего также находятся performance и partner_telecom

In [ ]:
channel_cohort = q("""
with params as 
(
    select '2026-08-30 00:00:00+00'::timestamptz as observation_end
)
, eligible_subs as 
(
    select
        s.subscription_id
        , s.owner_client_id
        , s.started_at
        , s.ended_at
        , c.acquisition_channel
    from prime.subscriptions s
    join prime.clients c
        on s.owner_client_id = c.client_id
    cross join params p
    where 1=1
    and s.started_at + interval '90 day' <= p.observation_end
)
, subscription_periods as 
(
    select
        s.subscription_id
        , s.acquisition_channel
        , date_trunc('quarter', s.started_at)::date as start_quarter
        , g.period_n + 1 as life_month
        , s.started_at + g.period_n * interval '30 day' as period_start
        , s.started_at + (g.period_n + 1) * interval '30 day' as period_end
        , s.ended_at
    from eligible_subs s
    cross join lateral generate_series(0, 2) as g(period_n)
    where 1=1
    and (
        s.ended_at is null
        or s.ended_at > s.started_at + g.period_n * interval '30 day'
    )
)
select
    start_quarter
    , acquisition_channel
    , count(*) as observations
    , avg(
        case
            when ended_at > period_start
            and ended_at <= period_end
            then 1.0
            else 0.0
        end
      ) * 100 as churn
from subscription_periods
where 1=1
group by start_quarter, acquisition_channel
order by start_quarter, acquisition_channel
""")

In [ ]:
cohort_pivot = channel_cohort.pivot(
    index="start_quarter"
    , columns="acquisition_channel"
    , values="churn"
)

cohort_pivot

In [ ]:
channel_cohort_full = channel_cohort[
    channel_cohort["start_quarter"] < pd.Timestamp("2026-04-01").date()
].copy()

In [ ]:
plt.figure(figsize=(11, 6))

for channel, data in channel_cohort_full.groupby("acquisition_channel"):
    if pd.isna(channel):
        continue

    plt.plot(
        data["start_quarter"]
        , data["churn"]
        , marker="o"
        , label=channel
    )

plt.title("Ранний отток по каналу и кварталу привлечения")
plt.xlabel("Квартал старта подписки")
plt.ylabel("Отток в первые 3 месяца, %")
plt.legend()

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Различия в оттоке между каналами примерно совпадают на всем рассматриваемом периоде. Во всех полных квартальных когортах максимальный отток наблюдается у social, затем у performance и partner_telecom. Минимальный у organic и referral. Следовательно, различия нельзя объяснить отдельным неудачным периодом привлечения.


## Экономика

In [ ]:
spend = q("""
    select
        channel
        , spend_date
        , amount
    from prime.marketing_spend
    where 1=1
""")

spend.head()

In [ ]:
cac_channel = q("""
with spend as 
(
    select
        channel
        , sum(amount) as spend
    from prime.marketing_spend
    where 1=1
    and spend_date >= '2025-02-01'
    and spend_date < '2026-04-01'
    group by channel
)
, registrations as 
(
    select
        acquisition_channel as channel
        , count(distinct client_id) as registrations
    from prime.clients
    where 1=1
    and registered_at >= '2025-02-01'
    and registered_at < '2026-04-01'
    and acquisition_channel is not null
    group by acquisition_channel
)
select
    s.channel
    , s.spend
    , r.registrations
    , s.spend / r.registrations as cac
from spend s
join registrations r
    on s.channel = r.channel
where 1=1
order by cac
""")

cac_channel

In [ ]:
early_channel_period = q("""
with params as 
(
    select '2026-08-30 00:00:00+00'::timestamptz as observation_end
)
, eligible_subs as 
(
    select
        s.subscription_id
        , s.started_at
        , s.ended_at
        , c.acquisition_channel
    from prime.subscriptions s
    join prime.clients c
        on s.owner_client_id = c.client_id
    cross join params p
    where 1=1
    and s.started_at >= '2025-02-01'
    and s.started_at < '2026-04-01'
    and s.started_at + interval '90 day' <= p.observation_end
    and c.acquisition_channel is not null
)
, periods as 
(
    select
        s.subscription_id
        , s.acquisition_channel
        , g.period_n + 1 as life_month
        , s.started_at + g.period_n * interval '30 day' as period_start
        , s.started_at + (g.period_n + 1) * interval '30 day' as period_end
        , s.ended_at
    from eligible_subs s
    cross join lateral generate_series(0, 2) as g(period_n)
    where 1=1
    and (
        s.ended_at is null
        or s.ended_at > s.started_at + g.period_n * interval '30 day'
    )
)
select
    acquisition_channel
    , count(*) as observations
    , avg(
        case
            when ended_at > period_start
            and ended_at <= period_end
            then 1.0
            else 0.0
        end
      ) * 100 as churn
from periods
where 1=1
group by acquisition_channel
order by churn desc
""")

early_channel_period

In [ ]:
channel_economics = early_channel_period.merge(
    cac_channel[
        ["channel", "cac"]
    ]
    , left_on="acquisition_channel"
    , right_on="channel"
    , how="inner"
    , validate="1:1"
)

channel_economics = (
    channel_economics
    .drop(columns="channel")
    .sort_values("cac")
)

channel_economics

In [ ]:
plt.figure(figsize=(9, 6))

plt.scatter(
    channel_economics["cac"]
    , channel_economics["churn"]
    , s=100
)

for _, row in channel_economics.iterrows():
    plt.annotate(
        row["acquisition_channel"]
        , (row["cac"], row["churn"])
        , xytext=(7, 5)
        , textcoords="offset points"
    )

plt.title("Стоимость привлечения и ранний отток по каналам")
plt.xlabel("CAC, ₽")
plt.ylabel("Отток в первые 3 месяца, %")

plt.tight_layout()
plt.show()

Наиболее эффективными являются organic и referral. У них низкая стоимость привлечения и низкий ранний отток. Performance имеет максимальный CAC (~1,5 тыс.) при оттоке около 5,5%, а social при CAC около 810 ₽ показывает максимальный отток (~7%). Таким образом, можно сказать, что высокая стоимость привлечения в ряде каналов не компенсируется более высоким качеством привлечённых клиентов.

In [ ]:
conversion_channel = q("""
with registered as 
(
    select
        client_id
        , acquisition_channel
        , registered_at
    from prime.clients
    where 1=1
    and registered_at >= '2025-02-01'
    and registered_at < '2026-04-01'
    and acquisition_channel is not null
)
, first_sub as 
(
    select
        owner_client_id as client_id
        , min(started_at) as first_sub_at
    from prime.subscriptions
    where 1=1
    group by owner_client_id
)
select
    r.acquisition_channel
    , count(*) as registrations
    , count(*) filter (
        where f.first_sub_at >= r.registered_at
        and f.first_sub_at < r.registered_at + interval '30 day'
      ) as subscribers_30
    , count(*) filter (
        where f.first_sub_at >= r.registered_at
        and f.first_sub_at < r.registered_at + interval '30 day'
      )::float / count(*) as conversion_30
from registered r
left join first_sub f
    on r.client_id = f.client_id
where 1=1
group by r.acquisition_channel
order by conversion_30 desc
""")

conversion_channel

In [ ]:
subscriber_cac = cac_channel.merge(
    conversion_channel[
        ["acquisition_channel", "subscribers_30"]
    ],
    left_on="channel",
    right_on="acquisition_channel",
    how="inner",
    validate="1:1"
)

subscriber_cac["subscriber_cac"] = (
    subscriber_cac["spend"]
    / subscriber_cac["subscribers_30"]
)

subscriber_cac[
    ["channel", "subscriber_cac"]
].sort_values("subscriber_cac")

In [ ]:
subscriber_economics = (
    early_channel_period
    .merge(
        subscriber_cac[
            ["channel", "subscriber_cac"]
        ],
        left_on="acquisition_channel",
        right_on="channel",
        how="inner",
        validate="1:1"
    )
    .drop(columns="channel")
)

subscriber_economics

In [ ]:
plt.figure(figsize=(9, 6))

plt.scatter(
    subscriber_economics["subscriber_cac"]
    , subscriber_economics["churn"]
    , s=120
)

for _, row in subscriber_economics.iterrows():
    plt.annotate(
        row["acquisition_channel"]
        , (row["subscriber_cac"], row["churn"])
        , xytext=(8, 6)
        , textcoords="offset points"
    )

plt.title("CAC подписчика и ранний отток по каналам")
plt.xlabel("CAC на подписчика, ₽")
plt.ylabel("Отток в первые 3 месяца, %")

plt.tight_layout()
plt.show()

In [ ]:
subscriber_economics[
    [
        "acquisition_channel",
        "subscriber_cac",
        "churn"
    ]
].sort_values("subscriber_cac")

Каналы существенно различаются по стоимости привлечения подписчика и по качеству привлечённой аудитории. Organic имеет минимальный CAC на подписчика и минимальный ранний отток. Performance имеет максимальный CAC при существенно более высоком оттоке, а social показывает самый высокий ранний отток при также высокой стоимости привлечения. Конверсия из регистрации в подписку между каналами почти одинакова, поэтому различия формируются в качестве и удержании подписчиков.

# факторная декомпозиция

In [ ]:
decomp = channel_cohort_full.copy()

decomp["year"] = pd.to_datetime(
    decomp["start_quarter"]
).dt.year

decomp = (
    decomp[
        decomp["year"].isin([2024, 2025])
    ]
    .groupby(
        ["year", "acquisition_channel"],
        as_index=False
    )
    .apply(
        lambda x: pd.Series({
            "observations": x["observations"].sum(),
            "churn": (
                (x["churn"] * x["observations"]).sum()
                / x["observations"].sum()
            )
        }),
        include_groups=False
    )
)

decomp

In [ ]:
decomp["share"] = (
    decomp["observations"]
    / decomp.groupby("year")["observations"].transform("sum")
)

decomp

In [ ]:
overall = (
    decomp
    .groupby("year")
    .apply(
        lambda x: (
            x["share"] * x["churn"]
        ).sum(),
        include_groups=False
    )
)

overall

In [ ]:
print(
    "Изменение общего churn:",
    round(overall.loc[2025] - overall.loc[2024], 3),
    "п.п."
)

In [ ]:
quarter_overall = (
    channel_cohort_full
    .groupby("start_quarter")
    .apply(
        lambda x: pd.Series({
            "observations": x["observations"].sum()
            , "churn": (
                (x["churn"] * x["observations"]).sum()
                / x["observations"].sum()
            )
        }),
        include_groups=False
    )
    .reset_index()
)

quarter_overall

In [ ]:
quarter_overall["change_pp"] = (
    quarter_overall["churn"].diff()
)

quarter_overall

In [ ]:
plt.figure(figsize=(10, 4))

plt.plot(
    quarter_overall["start_quarter"]
    , quarter_overall["churn"]
    , marker="o"
)

plt.title("Общий ранний отток по квартальным когортам")
plt.xlabel("Квартал старта подписки")
plt.ylabel("Отток в первые 3 месяца, %")

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
quarter_overall["change_pp"] = quarter_overall["churn"].diff()

quarter_overall[
    ["start_quarter", "churn", "change_pp"]
]

In [ ]:
biggest_change = (
    quarter_overall
    .dropna()
    .loc[
        quarter_overall["change_pp"].abs().idxmax()
    ]
)

biggest_change

In [ ]:
period_before = pd.Timestamp("2024-07-01").date()
period_after = pd.Timestamp("2024-10-01").date()

before = channel_cohort_full[
    channel_cohort_full["start_quarter"] == period_before
].copy()

after = channel_cohort_full[
    channel_cohort_full["start_quarter"] == period_after
].copy()

In [ ]:
before["share"] = (
    before["observations"]
    / before["observations"].sum()
)

after["share"] = (
    after["observations"]
    / after["observations"].sum()
)

In [ ]:
decomp = before[
    ["acquisition_channel", "share", "churn"]
].merge(
    after[
        ["acquisition_channel", "share", "churn"]
    ],
    on="acquisition_channel",
    how="inner",
    suffixes=("_before", "_after"),
    validate="1:1"
)

decomp

In [ ]:
decomp["mix_effect"] = (
    (decomp["share_after"] - decomp["share_before"])
    * decomp["churn_before"]
)

decomp["churn_effect"] = (
    decomp["share_before"]
    * (decomp["churn_after"] - decomp["churn_before"])
)

decomp["interaction"] = (
    (decomp["share_after"] - decomp["share_before"])
    * (decomp["churn_after"] - decomp["churn_before"])
)

decomp["total_effect"] = (
    decomp["mix_effect"]
    + decomp["churn_effect"]
    + decomp["interaction"]
)

In [ ]:
effects = decomp[
    [
        "mix_effect",
        "churn_effect",
        "interaction",
        "total_effect"
    ]
].sum()

effects

In [ ]:
before_total = (
    decomp["share_before"]
    * decomp["churn_before"]
).sum()

after_total = (
    decomp["share_after"]
    * decomp["churn_after"]
).sum()

print(
    "Факт:",
    round(after_total - before_total, 6),
    "п.п."
)

print(
    "Декомпозиция:",
    round(effects["total_effect"], 6),
    "п.п."
)

In [ ]:
decomp[
    [
        "acquisition_channel",
        "mix_effect",
        "churn_effect",
        "interaction",
        "total_effect"
    ]
].sort_values("total_effect")

In [ ]:
plot_decomp = decomp.copy()

plot_decomp["acquisition_channel"] = (
    plot_decomp["acquisition_channel"]
    .fillna("unknown")
)

plot_decomp = (
    plot_decomp
    .sort_values("total_effect", ascending=False)
    .reset_index(drop=True)
)

fig, ax = plt.subplots(figsize=(10, 5.5))

bars = ax.barh(
    plot_decomp["acquisition_channel"]
    , plot_decomp["total_effect"]
)

ax.axvline(0, linewidth=1)

for bar, value in zip(
    bars,
    plot_decomp["total_effect"]
):
    y = bar.get_y() + bar.get_height() / 2

    if value < -0.01:
        # крупные отрицательные — подпись внутри столбца
        ax.text(
            value + 0.0015
            , y
            , f"{value:.3f} п.п."
            , va="center"
            , ha="left"
        )
    elif value < 0:
        # маленькие отрицательные — рядом со столбцом
        ax.text(
            value - 0.001
            , y
            , f"{value:.3f} п.п."
            , va="center"
            , ha="right"
        )
    else:
        ax.text(
            value + 0.0005
            , y
            , f"+{value:.3f} п.п."
            , va="center"
            , ha="left"
        )

ax.set_title(
    "Вклад каналов в изменение раннего оттока\n"
    "2024-Q3 → 2024-Q4"
)

ax.set_xlabel("Вклад в изменение оттока, п.п.")
ax.set_ylabel("")

ax.grid(
    axis="x",
    alpha=0.2
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.set_xlim(
    plot_decomp["total_effect"].min() - 0.006,
    plot_decomp["total_effect"].max() + 0.012
)

plt.tight_layout()
plt.show()

Между 2024-Q3 и 2024-Q4 ранний отток снизился на 0,161 п.п. Наибольший вклад в снижение дали performance, social и crm. Partner_telecom, напротив, частично компенсировал улучшение. Основная часть снижения связана с уменьшением оттока внутри каналов